In [63]:
# this notebook is going to be the third and last step
# where I run inference on detection, save the crops, and track them
# then feed it into vit
# it is a pretty rough setup of things but ill refine it later

In [64]:
# ------ Imports ------
from ultralytics import YOLO
import supervision as sv
import cv2
import os
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from collections import defaultdict, deque
import torch
import torchvision.transforms as T
import re

In [65]:
# ------ CONFIGS ------
IMAGE_DIR = "data/images"
FPS = 10  # Adjust to match your original video FPS
CLASS_NAMES = {0: "Bee"}
BUFFER_SIZE = 16  # Number of frames per clip — for ViT feeding
OUTPUT_DIR = "data/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ------ INITS ------
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)
delay = int(1000 / FPS)
images = sorted(
    [f for f in os.listdir(IMAGE_DIR) if f.endswith((".jpg", ".png"))],
    key=lambda x: [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', x)]
)
track_buffers = defaultdict(lambda: deque(maxlen=BUFFER_SIZE)) # ViT frame buffer - every IDed bee and how many frames they have

In [66]:
# ------ SAHI WRAPPER ------
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path='best.pt',  # or full path to best.pt
    confidence_threshold=0.35,
    device='cuda:0'
)

In [67]:
# ------ UTILITY FUNCTION(S) ------

# Extracts a crop from a frame with padding around the bounding box
def extract_crop(frame, xyxy, padding=0.35):
    """
    Crop a detection from a frame with percentage-based padding.
    
    Args:
        frame:   The full image (H, W, C) numpy array
        xyxy:    Bounding box as [x1, y1, x2, y2]
        padding: Fractional padding to add around the box (0.25 = 25%)
    
    Returns:
        crop: The cropped region as a numpy array, or None if invalid
    """
    frame_h, frame_w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, xyxy)
    
    box_w = x2 - x1
    box_h = y2 - y1
    
    pad_x = int(box_w * padding)
    pad_y = int(box_h * padding)
    
    # Expand and clamp to frame boundaries
    x1_p = max(0, x1 - pad_x)
    y1_p = max(0, y1 - pad_y)
    x2_p = min(frame_w, x2 + pad_x)
    y2_p = min(frame_h, y2 + pad_y)
    
    crop = frame[y1_p:y2_p, x1_p:x2_p]
    
    # Guard against degenerate boxes
    if crop.size == 0:
        return None
    
    return crop


# Standard ImageNet normalization — used by most pretrained ViT backbones
preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),                                # converts to [C, H, W], scales to [0, 1]
    T.Normalize(mean=[0.485, 0.456, 0.406],     # ImageNet mean
                std=[0.229, 0.224, 0.225])       # ImageNet std
])

def prepare_clip(buffer):
    """
    Takes a full buffer of crops and returns a ViT-ready tensor.
    
    Args:
        buffer: deque of crops (numpy arrays in BGR format from OpenCV)
    
    Returns:
        Tensor of shape [T, C, H, W] or None if buffer isn't full yet
    """
    if len(buffer) < BUFFER_SIZE:
        return None  # Not ready yet
    
    frames = []
    for crop in buffer:
        # OpenCV loads as BGR, convert to RGB first
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        tensor = preprocess(crop_rgb)   # [C, H, W]
        frames.append(tensor)
    
    clip = torch.stack(frames)  # [T, C, H, W]
    return clip

In [68]:
# ------ SANITY CHECKS ------ Commented

### extract_crop sanity check ###
# test_img_path = os.path.join(IMAGE_DIR, images[0])
# test_img_path = "1.jpg"
# test_frame = cv2.imread(test_img_path)
# test_result = get_sliced_prediction(
#     test_img_path, sahi_model,
#     slice_height=640, slice_width=640,
#     overlap_height_ratio=0.25, overlap_width_ratio=0.25,
#     verbose=0
# )
# for i, pred in enumerate(test_result.object_prediction_list):
#     bbox = pred.bbox.to_xyxy()
#     crop = extract_crop(test_frame, bbox, padding=0.25)
#     if crop is not None:
#         print(f"Detection {i}: box={[int(b) for b in bbox]}, crop shape={crop.shape}")
#         cv2.imshow(f"Crop {i}", crop)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

### Buffer sanity check ###
# track_id_to_check = list(track_buffers.keys())[6]  # just grabs the first one
# buffer = track_buffers[track_id_to_check]
# print(f"Showing crops for Track #{track_id_to_check}")
# for i, crop in enumerate(buffer):
#     cv2.imshow(f"Track #{track_id_to_check} | Frame {i}", crop)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [70]:
# ------ MAIN LOOP ------
for frame_idx, img_name in enumerate(images[:5]):
    
    img_path = os.path.join(IMAGE_DIR, img_name)
    frame = cv2.imread(img_path)
    
    if frame is None:
        continue
    
    result = get_sliced_prediction( 
        img_path,
        sahi_model,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.25,
        overlap_width_ratio=0.25,
        verbose=0
    )
    
    boxes = []
    class_ids = []
    confidences = []
    
    for pred in result.object_prediction_list:
        bbox = pred.bbox.to_xyxy()
        boxes.append(bbox)
        class_ids.append(pred.category.id)
        confidences.append(pred.score.value)
    
    if len(boxes) > 0:
        detections = sv.Detections(
            xyxy=np.array(boxes),
            class_id=np.array(class_ids),
            confidence=np.array(confidences)
        )
    else:
        detections = sv.Detections.empty()
    
    print(f"Frame {frame_idx} | {img_name} | Detections: {len(detections)}")
    
    for i in range(len(detections)):
        crop = extract_crop(frame, detections.xyxy[i], padding=0.35)
        if crop is not None:
            track_buffers[i].append(crop)  # keyed by detection index within frame
    
    # Labels with just confidence
    labels = [f"{confidences[i]:.2f}" for i in range(len(detections))]
    
    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
    
    cv2.putText(annotated_frame, f"Frame: {frame_idx} | {img_name} | Detections: {len(detections)}", 
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    out_path = os.path.join(OUTPUT_DIR, img_name)
    cv2.imwrite(out_path, annotated_frame)
    
    cv2.imshow("YOLO + SAHI Detections", annotated_frame)
    
    key = cv2.waitKey(delay)
    if key == 27:
        break
    elif key == 32:
        cv2.waitKey(0)

cv2.destroyAllWindows()

Frame 0 | 20230609a48.jpg | Detections: 5
Frame 1 | 20230609a54.jpg | Detections: 5
Frame 2 | 20230609a60.jpg | Detections: 4
Frame 3 | 20230609a66.jpg | Detections: 6
Frame 4 | 20230609a72.jpg | Detections: 4


In [24]:
# ------ ViT PART ------

# Standard ImageNet normalization — used by most pretrained ViT backbones
preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),                                # converts to [C, H, W], scales to [0, 1]
    T.Normalize(mean=[0.485, 0.456, 0.406],     # ImageNet mean
                std=[0.229, 0.224, 0.225])       # ImageNet std
])